# Training Language Classifier

Route user text to **Arabic** or **English** with classical NLP (char TF-IDF + Linear SVM).

## 1. Imports

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

ROOT = Path("..") if (Path("..") / "data").exists() else Path(".")
sys.path.insert(0, str(ROOT.resolve()))

from src.labels import ARABIC, ENGLISH
from src.language_model import (
    LANG_TEXT_MAX_CHARS,
    LanguageClassifier,
    make_language_pipeline,
    script_counts,
)
from src.Preprocessing_pipeline import preprocess_language_detection

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", "{:.4f}".format)
LANG_LABELS = [ARABIC, ENGLISH]


## 2. Load Data

Balanced sample: `min(n_ar, n_en)` from each corpus so the detector is not skewed by IMDb size.

In [2]:
en = pd.read_csv(ROOT / "data" / "raw" / "english" / "MovieReviewTrainingDatabase.csv")["review"]
ar = pd.read_csv(ROOT / "data" / "raw" / "arabic" / "Final_Data.csv")["review_description"]
en = en.dropna().astype(str)
ar = ar.dropna().astype(str)
en = en[en.str.strip().str.len() > 0].drop_duplicates()
ar = ar[ar.str.strip().str.len() > 0].drop_duplicates()
print("English pool", len(en), "| Arabic pool", len(ar))


English pool 24904 | Arabic pool 39003


## 3. EDA

In [3]:
print("English char length:\n", en.str.len().describe())
print("Arabic char length:\n", ar.str.len().describe())


English char length:
 count   24904.0000
mean     1306.1254
std       988.4471
min        52.0000
25%       695.0000
50%       966.0000
75%      1588.0000
max     13603.0000
Name: review, dtype: float64
Arabic char length:
 count   39003.0000
mean       51.9400
std        68.7915
min         1.0000
25%        15.0000
50%        29.0000
75%        62.0000
max      2481.0000
Name: review_description, dtype: float64


## 4. Cleaning

Truncate to 120 characters so IMDb length is not a cheap language cue. Then equal class counts.

In [4]:
n = int(min(len(en), len(ar)))
lang_df = pd.concat(
    [
        pd.DataFrame({"text": en.sample(n, random_state=42).str.slice(0, LANG_TEXT_MAX_CHARS).values, "language": ENGLISH}),
        pd.DataFrame({"text": ar.sample(n, random_state=42).str.slice(0, LANG_TEXT_MAX_CHARS).values, "language": ARABIC}),
    ],
    ignore_index=True,
)
print(lang_df["language"].value_counts())


language
English    24904
Arabic     24904
Name: count, dtype: int64


## 5. Preprocessing

Language-agnostic: URLs/HTML off, keep letters of any script. No English stopwords, no Alef folding.

In [5]:
lang_df["clean_text"] = lang_df["text"].map(preprocess_language_detection)
lang_df = lang_df[lang_df["clean_text"].str.len() > 0]
n_bal = int(lang_df["language"].value_counts().min())
lang_df = (
    lang_df.groupby("language", group_keys=False)
    .sample(n_bal, random_state=42)
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)
print(lang_df["language"].value_counts())
lang_df[["text", "language"]].head(4)


language
Arabic     24365
English    24365
Name: count, dtype: int64


,text,language
0,نصب ف نصب اكتب الكود مش بيرده يدخل,Arabic
1,كان جيد ببدايته حاليا ما بصير اقيم اي طلب بوصلني ولا احكي رأيي باي مطعم على ...,Arabic
2,رايع جدا جدا,Arabic
3,غير منسب,Arabic


## 6. Train/Test Split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    lang_df["clean_text"], lang_df["language"], test_size=0.2, random_state=42, stratify=lang_df["language"]
)
print("train", len(X_train), "test", len(X_test))


train 38984 test 9746


## 7. Feature Extraction

Character TF-IDF `analyzer='char', ngram_range=(2, 5)`: English `th`/`ing` vs Arabic `ال`/`ية`.

## 8. Baseline Model

Char TF-IDF + Linear SVM (the recommended classical detector).

In [7]:
def scores(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

language_pipeline = make_language_pipeline()
language_pipeline.fit(X_train, y_train)
y_pred = language_pipeline.predict(X_test)
print(language_pipeline)
print(classification_report(y_test, y_pred, labels=LANG_LABELS, digits=4))
print(pd.Series(scores(y_test, y_pred), name="test"))


Pipeline(steps=[('tfidf',
                 TfidfVectorizer(analyzer='char', max_features=40000,
                                 ngram_range=(2, 5))),
                ('clf', LinearSVC())])
              precision    recall  f1-score   support

      Arabic     0.9998    0.9977    0.9988      4873
     English     0.9977    0.9998    0.9988      4873

    accuracy                         0.9988      9746
   macro avg     0.9988    0.9988    0.9988      9746
weighted avg     0.9988    0.9988    0.9988      9746

Accuracy    0.9988
Precision   0.9988
Recall      0.9988
F1          0.9988
Name: test, dtype: float64


## 9. Model Experiments

Char n-grams already separate the scripts almost perfectly. No extra grid: this is the production config.

## 10. Evaluation

In [8]:
cm = confusion_matrix(y_test, y_pred, labels=LANG_LABELS)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=LANG_LABELS).plot(cmap="Blues", ax=ax, colorbar=False)
ax.set_title("Language detector — test confusion matrix")
plt.tight_layout()
plt.show()


/tmp/ipykernel_56983/879367332.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. Best Model

In [9]:
print(language_pipeline)
print("test F1", scores(y_test, y_pred)["F1"])


Pipeline(steps=[('tfidf',
                 TfidfVectorizer(analyzer='char', max_features=40000,
                                 ngram_range=(2, 5))),
                ('clf', LinearSVC())])


test F1 0.9987687243347372


## 12. Save Model

In [10]:
clf = LanguageClassifier(language_pipeline)
path = clf.save(ROOT / "models" / "Language_classifier_weights.pkl")
print("saved", path.resolve())


saved /home/abdlrhman/Courses/IEEE SSCS/NLP/Week1/Project1/models/Language_classifier_weights.pkl


## 13. Example Predictions

Mixed-language policy (not in the PDF): both alphabets → **Arabic**. Latin only → English. Digits/emoji → English.

In [11]:
model = LanguageClassifier.load(path)
cases = [
    "good", "كويس", "12345", "😂😂😂",
    "hello يا جماعة", "AI جميل جدا", "iPhone ممتاز",
    "تمام", "ok", "Hello world", "مرحبا بالعالم",
    "I absolutely loved this movie.",
    "الخدمة ممتازة والتجربة كانت رائعة",
]
print(f"{'text':32} {'ar':>3} {'la':>3}  pred")
for text in cases:
    n_ar, n_la = script_counts(text)
    print(f"{text:32} {n_ar:3d} {n_la:3d}  {model.predict(text)}")


text                              ar  la  pred
good                               0   4  English
كويس                               4   0  Arabic
12345                              0   0  English
😂😂😂                                0   0  English
hello يا جماعة                     7   5  Arabic
AI جميل جدا                        7   2  Arabic
iPhone ممتاز                       5   6  Arabic
تمام                               4   0  Arabic
ok                                 0   2  English
Hello world                        0  10  English
مرحبا بالعالم                     12   0  Arabic
I absolutely loved this movie.     0  25  English
الخدمة ممتازة والتجربة كانت رائعة  29   0  Arabic


## Integration check (master pipeline)

In [12]:
from src.pipeline import NeurovaNLPPipeline

nlp = NeurovaNLPPipeline.load()
for text in (
    "I absolutely loved this movie.",
    "This was one of the worst movies I've ever watched.",
    "الخدمة ممتازة والتجربة كانت رائعة",
    "الخدمة سيئة جدا ولن أكرر التجربة",
):
    print(nlp.analyze(text))


{'User Text': 'I absolutely loved this movie.', 'Language': 'English', 'Sentiment Classification': 'Positive'}
{'User Text': "This was one of the worst movies I've ever watched.", 'Language': 'English', 'Sentiment Classification': 'Negative'}
{'User Text': 'الخدمة ممتازة والتجربة كانت رائعة', 'Language': 'Arabic', 'Sentiment Classification': 'Positive'}
{'User Text': 'الخدمة سيئة جدا ولن أكرر التجربة', 'Language': 'Arabic', 'Sentiment Classification': 'Negative'}
